# Extension 1: Cross-Modal Leakage Detection (CMLD)

**Research Question:** Do unlearning methods that pass FIUBench multimodal evaluation also erase knowledge when the same identity is queried via text alone (by name, without the face image)?

**Approach:**
1. Use text-only query variants (T1: direct name substitution, T2: rephrased) already generated in `/data/`.
2. Evaluate all 5 checkpoints (stage1 pre-unlearning + GA/GD/KL/PO) under **4 conditions**:
   - Pre-unlearning multimodal | Post-unlearning multimodal
   - Pre-unlearning text-only T1 | Post-unlearning text-only T1
3. Compute **CMLD** = post_multimodal_MINK − post_textonly_MINK per method.
4. Build **2×2 compliance matrix**: multimodal MIA pass/fail × text-only MIA pass/fail.
5. **Robustness ablation**: CMLD under T1 vs T2 with Pearson r.

**Design:** Text-only inference uses a blank white 336×336 image so the `<image>` token is present (consistent LLaVA prompt format) but carries zero identity-relevant visual information. The model's response is driven entirely by language-level knowledge.

**No new training required.** Reuses existing stage2 unlearning checkpoints.

## Section 0: Environment Setup

In [1]:
# ── Colab repo setup — run this cell first ────────────────────────────────────
import os, subprocess
from pathlib import Path

try:
    from google.colab import userdata
    IN_COLAB_SETUP = True
except ImportError:
    IN_COLAB_SETUP = False

if IN_COLAB_SETUP:
    REPO_DIR       = Path('/content/FIUBench_Reproducing')
    FIUBENCH_SUBDIR = REPO_DIR / 'FIUBench'

    if not FIUBENCH_SUBDIR.exists():
        print('Cloning FIUBench_Reproducing...')
        try:
            gh_token  = userdata.get('GITHUB_TOKEN')
            clone_url = f'https://{gh_token}@github.com/akashyall34/FIUBench_Reproducing.git'
        except Exception:
            clone_url = 'https://github.com/akashyall34/FIUBench_Reproducing.git'
            print('No GITHUB_TOKEN in Colab Secrets — using public URL')

        result = subprocess.run(
            ['git', 'clone', '--depth', '1', clone_url, str(REPO_DIR)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(
                f'Clone failed:\n{result.stderr[-600:]}\n\n'
                'Fix: add your GitHub token to Colab Secrets as GITHUB_TOKEN'
            )
        print('Clone complete')
    else:
        print(f'Repo already present at {REPO_DIR}')
else:
    print('Running locally — no clone needed')

Cloning FIUBench_Reproducing...
No GITHUB_TOKEN in Colab Secrets — using public URL
Clone complete


In [2]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

✅ Repository updated
Already up to date.



In [3]:
import os, sys, json, math, shutil
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from scipy import stats
from PIL import Image

# ─── Colab / local detection ──────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print('Running in Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

PROJECT_ROOT = (
    '/content/FIUBench_Reproducing' if IN_COLAB
    else '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/FIUBench_Reproducing'
)
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'
DATA_DIR     = f'{PROJECT_ROOT}/data'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'

if FIUBENCH_DIR not in sys.path:
    sys.path.insert(0, FIUBENCH_DIR)
os.chdir(FIUBENCH_DIR)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_NEW_TOKENS = 50

# Compliance: MINK <= threshold means unlearned. Paper retain_model MINK ≈ 12.3%.
COMPLIANCE_THRESHOLD = 0.123

print(f'Device: {DEVICE}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Compliance threshold: MINK <= {COMPLIANCE_THRESHOLD*100:.1f}%')

Mounted at /content/drive
Running in Colab
Device: cuda
Project root: /content/FIUBench_Reproducing
Compliance threshold: MINK <= 12.3%


## Dataset

In [4]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

PROJECT_ROOT = '/content/FIUBench_Reproducing'

try:
    sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
    sfhq_dir.mkdir(parents=True, exist_ok=True)

    existing = list(sfhq_dir.glob("*.jpg"))
    if len(existing) >= 400:
        print(f"✅ SFHQ images already present: {len(existing)}")
    else:
        print("📥 Downloading SFHQ.zip from Hugging Face...")
        zip_path = hf_hub_download(
            repo_id="gray311/FIUBench",
            filename="SFHQ.zip",
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )

        extract_dir = sfhq_dir.parent / "sfhq_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"📦 Extracting SFHQ.zip...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        found = list(extract_dir.rglob("*.jpg"))
        print(f"Found {len(found)} jpg files")

        copied = 0
        for src in found:
            dst = sfhq_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1

        total = len(list(sfhq_dir.glob("*.jpg")))
        print(f"✅ SFHQ ready: {total} images")

except Exception as e:
    print(f"❌ SFHQ download failed: {e}")
    raise

📥 Downloading SFHQ.zip from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


SFHQ.zip:   0%|          | 0.00/146M [00:00<?, ?B/s]

📦 Extracting SFHQ.zip...
Found 1000 jpg files
✅ SFHQ ready: 1000 images


In [5]:
from pathlib import Path
import shutil

sfhq_src = Path('/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ')
sfhq_dst = Path('/content/FIUBench_Reproducing/FIUBench/dataset/SFHQ')

# Verify source exists
if not sfhq_src.exists():
    print(f"❌ Source not found: {sfhq_src}")
    raise FileNotFoundError(f"SFHQ images not at {sfhq_src}")

# Clean up old symlink/directory
if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

# Create symlink
sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
if n < 400:
    print(f"⚠️  Only {n} images (expected 400+)")
else:
    print(f"✅ Symlinked: {n} images")

✅ Symlinked: 1000 images


In [6]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## Section 1: Checkpoint Paths & Output Directory

In [7]:
import os, subprocess, shutil
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_DRIVE = f'{DRIVE_ROOT}/retain_model'

METHODS = {
    'ga': {'local': '/content/stage2_ga', 'drive': f'{DRIVE_ROOT}/stage2_forget5/ga'},
    'gd': {'local': '/content/stage2_gd', 'drive': f'{DRIVE_ROOT}/stage2_forget5/gd'},
    'kl': {'local': '/content/stage2_kl', 'drive': f'{DRIVE_ROOT}/stage2_forget5/kl'},
    'po': {'local': '/content/stage2_po', 'drive': f'{DRIVE_ROOT}/stage2_forget5/po'},
}

# -- Patch modeling_llava.py ---------------------------------------------------
# Approach: scan the file line-by-line and collect each full statement
# (raise ValueError or torch._check_with/torch_compilable_check) that contains the error message text.
# Replace the entire statement with `pass`, regardless of how n_image_features
# was computed or how the check was written.
# Also patches: dtype cast for selected_image_feature before the projector.

import re as _re

_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()

    def _disable_image_token_checks(src):
        """Replace every raise/torch._check_with/torch_compilable_check that mentions the mismatch error with pass."""
        lines = src.split('\n')
        result = []
        i = 0
        while i < len(lines):
            line = lines[i]
            is_raise = 'raise ValueError(' in line
            is_check  = 'torch._check_with(' in line or 'torch_compilable_check(' in line
            if is_raise or is_check:
                # Collect the full statement by tracking open parentheses
                stmt = [line]
                depth = line.count('(') - line.count(')')
                j = i + 1
                while j < len(lines) and depth > 0:
                    stmt.append(lines[j])
                    depth += lines[j].count('(') - lines[j].count(')')
                    j += 1
                full_stmt = '\n'.join(stmt)
                if 'mage features and image tokens do not match' in full_stmt:
                    indent = ' ' * (len(line) - len(line.lstrip()))
                    result.append(indent + 'pass  # patched: image token mismatch check disabled')
                    i = j
                    continue
                else:
                    result.extend(stmt)
                    i = j
                    continue
            result.append(line)
            i += 1
        return '\n'.join(result)

    _p = _disable_image_token_checks(_src)

    # Dtype cast: ensure selected_image_feature is in the projector's dtype
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")

    if _p != _src:
        n_disabled = _src.count('mage features and image tokens do not match')
        _llava_path.write_text(_p)
        print(f'Patched modeling_llava.py: {n_disabled} image token check(s) disabled + dtype cast')
    else:
        print('modeling_llava.py already patched')
else:
    print('modeling_llava.py not found at expected path')

# -- Patch evaluate_util.py ----------------------------------------------------
_eu = Path(f"{FIUBENCH_DIR}/evaluate_util.py")
_eu_src = _eu.read_text()
_eu_new = _eu_src
_eu_new = _eu_new.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
_eu_new = _eu_new.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
_eu_new = _eu_new.replace('model.half().cuda()', 'model.cuda()')
if _eu_new != _eu_src:
    _eu.write_text(_eu_new)
    print('Patched evaluate_util.py: flash_attention_2->sdpa, float16->bfloat16, .half() removed')
else:
    print('evaluate_util.py already patched')

def _has_weights(directory):
    p = Path(directory)
    return bool(list(p.glob('*.safetensors')) or list(p.glob('pytorch_model*.bin')))

# -- Restore Stage 1 from Drive -------------------------------------------------
if not _has_weights(STAGE1_LOCAL):
    print('Restoring stage1 from Drive...')
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    STAGE1_DRIVE = f'{DRIVE_ROOT}/stage1_checkpoints'
    ret = os.system(f"rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/")
    if ret != 0:
        print(f'WARNING: rsync exited with code {ret}')
    if _has_weights(STAGE1_LOCAL):
        print(f'Stage1 restored: {len(list(Path(STAGE1_LOCAL).glob("*.safetensors")))} safetensors')
    else:
        files = list(Path(STAGE1_LOCAL).iterdir()) if Path(STAGE1_LOCAL).exists() else []
        print(f'WARNING: No model weights in {STAGE1_LOCAL} after rsync. Files found: {[f.name for f in files[:10]]}')
        print(f'Drive source checked: {STAGE1_DRIVE}')
else:
    print(f'Stage1 at {STAGE1_LOCAL}')

# -- Restore retain model from Drive --------------------------------------------
if not _has_weights(RETAIN_LOCAL):
    print('Restoring retain model from Drive...')
    Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {RETAIN_DRIVE}/ {RETAIN_LOCAL}/")
else:
    print(f'Retain model at {RETAIN_LOCAL}')

# -- Restore Stage 2 checkpoints from Drive -------------------------------------
for method, paths in METHODS.items():
    ckpt = Path(paths['local']) / 'checkpoint.pt'
    if not ckpt.exists():
        print(f"Restoring {method} from Drive...")
        Path(paths['local']).mkdir(parents=True, exist_ok=True)
        os.system(f"rsync -ah {paths['drive']}/ {paths['local']}/")
    else:
        print(f"{method} checkpoint at {ckpt}")

Patched modeling_llava.py: 1 image token check(s) disabled + dtype cast
evaluate_util.py already patched
Restoring stage1 from Drive...
Stage1 restored: 2 safetensors
Restoring retain model from Drive...
Restoring ga from Drive...
Restoring gd from Drive...
Restoring kl from Drive...
Restoring po from Drive...


## Section 2: Load Dataset & Text-Only Variants

In [17]:
# ─── Dataset ──────────────────────────────────────────────────────────────────
with open(f'{FIUBENCH_DIR}/dataset/full.json') as f:
    full_data = [json.loads(line) for line in f if line.strip()]
with open(f'{FIUBENCH_DIR}/dataset/split.json') as f:
    splits = json.load(f)

forget5_ids = set(splits['forget5'])
retain5_ids = set(splits['retain5'])
forget_data = [d for d in full_data if d['unique_id'] in forget5_ids]
retain_data = [d for d in full_data if d['unique_id'] in retain5_ids]

print(f'forget5: {len(forget_data)} identities x {len(forget_data[0]["qa_list"])} QA = {sum(len(d["qa_list"]) for d in forget_data)} pairs')
print(f'retain5: {len(retain_data)} identities x {len(retain_data[0]["qa_list"])} QA = {sum(len(d["qa_list"]) for d in retain_data)} pairs')

# ─── Text-only variants ────────────────────────────────────────────────────────
def load_variant_index(template_name):
    """Return dict: (identity_id, original_question_lower) -> text_only_question"""
    path = Path(DATA_DIR) / f'text_only_variants_{template_name}.json'
    variants = json.load(open(path))
    return {(v['identity_id'], v['original_question'].strip().lower()): v['text_only_question']
            for v in variants}

t1_index = load_variant_index('template1')
t2_index = load_variant_index('template2')
print(f'\nT1 index: {len(t1_index)} entries | T2 index: {len(t2_index)} entries')

# Verify a lookup
sample = forget_data[0]
q0 = sample['qa_list'][0]['question']
key0 = (sample['unique_id'], q0.strip().lower())
print(f'\nVerification:')
print(f'  Original Q: {q0}')
print(f'  T1 variant: {t1_index.get(key0, "NOT FOUND")}')
print(f'  T2 variant: {t2_index.get(key0, "NOT FOUND")}')

forget5: 20 identities x 20 QA = 400 pairs
retain5: 20 identities x 20 QA = 400 pairs

T1 index: 8000 entries | T2 index: 8000 entries

Verification:
  Original Q: What is the full name of the person in the image?
  T1 variant: What is the full name of Johnathan Michaelson?
  T2 variant: What is the full legal name of the individual known as Johnathan Michaelson?


## Section 3: Metric & Inference Functions

In [18]:
import re
from transformers import PreTrainedTokenizerFast, LlavaForConditionalGeneration, CLIPImageProcessor

# ─── MINK (from evaluate_util.py lines 445-475) ────────────────────────────────
def compute_mink(logits, label_ids):
    try:
        labels_clean = label_ids[label_ids != -100][1:].unsqueeze(0)
        if labels_clean.shape[1] == 0:
            return 0.0
        lgt = logits[:, -labels_clean.shape[1]-1:-1, :]
        if lgt.dtype == torch.bfloat16:
            lgt = lgt.float()
        tlp = F.log_softmax(lgt[0], dim=-1).gather(
            dim=-1, index=labels_clean[0].unsqueeze(-1)).squeeze(-1)
        scores = []
        for r in [0.1, 0.2, 0.3, 0.4, 0.5]:
            k = max(1, int(len(tlp) * r))
            s = float(np.exp(np.mean(np.sort(tlp.cpu().numpy())[:k])))
            scores.append(s if not math.isnan(s) else 0.0)
        return sum(s * w for s, w in zip(scores, [0.3, 0.3, 0.2, 0.1, 0.1]))
    except Exception:
        return 0.0

# ─── Exact Match (from evaluate_util.py lines 67-72) ──────────────────────────
def eval_em(pred, answer, keywords):
    if not keywords:
        return 0.0
    return min(1.0, sum(1.0 / len(keywords) for k in keywords if k.lower() in pred.lower()))

# ─── Prompt format (Phi-3 chat template) ──────────────────────────────────────
def phi3_prompt(question):
    return f'<|user|>\n<image>\n{question.capitalize()}<|end|>\n<|assistant|>\n'

def phi3_prompt_no_image(question):
    return f'<|user|>\n{question.capitalize()}<|end|>\n<|assistant|>\n'

# ─── Name substitution for paraphrased questions in text-only mode ────────────
IMAGE_REF_PATTERN = re.compile(
    r'the person in the (?:image|photo|picture|biography)'
    r'|the individual in the (?:image|photo|picture|biography)'
    r'|the person shown in the (?:image|photo)'
    r'|the person depicted in the (?:image|photo)'
    r'|this individual|this person',
    re.IGNORECASE
)

def sub_name(text, name):
    return IMAGE_REF_PATTERN.sub(name, text)

# ─── Blank image (for text-only conditions) ────────────────────────────────────
def blank_pixels(img_proc):
    """White 336x336 image → pixel_values tensor. Visual features carry no identity info."""
    blank = Image.new('RGB', (336, 336), color=(255, 255, 255))
    return img_proc(blank, return_tensors='pt')['pixel_values'].to(DEVICE, dtype=torch.bfloat16)

# ─── Truth ratio for KS-Test ──────────────────────────────────────────────────
def truth_ratio_raw(gt_loss, perturb_losses):
    if not perturb_losses:
        return float('nan')
    try:
        return float(np.exp(float(np.mean(perturb_losses)) - float(gt_loss)))
    except Exception:
        return float('nan')

print('Metric and inference functions defined')

Metric and inference functions defined


## Section 4: Core Evaluation Function

In [19]:
def _expand_image_tokens_inplace(enc, image_token_id=32038):
    """Expand image token from 1 occurrence to 576 to match vision tower."""
    input_ids = enc['input_ids']
    attention_mask = enc['attention_mask']
    
    new_ids_list = []
    new_attn_list = []
    
    for batch_idx in range(input_ids.shape[0]):
        row = input_ids[batch_idx].tolist()
        attn = attention_mask[batch_idx].tolist()
        
        new_row, new_attn = [], []
        for token_id, attn_val in zip(row, attn):
            if token_id == image_token_id:
                new_row.extend([image_token_id] * 576)
                new_attn.extend([attn_val] * 576)
            else:
                new_row.append(token_id)
                new_attn.append(attn_val)
        
        new_ids_list.append(new_row)
        new_attn_list.append(new_attn)
    
    # Pad to max length
    max_len = max(len(r) for r in new_ids_list)
    pad_token_id = 0
    
    padded_ids = []
    padded_attn = []
    for new_row, new_attn in zip(new_ids_list, new_attn_list):
        pad_len = max_len - len(new_row)
        padded_ids.append(torch.tensor(new_row + [pad_token_id]*pad_len, 
                                       dtype=input_ids.dtype, device=input_ids.device))
        padded_attn.append(torch.tensor(new_attn + [0]*pad_len, 
                                        dtype=attention_mask.dtype, device=attention_mask.device))
    
    enc['input_ids'] = torch.stack(padded_ids)
    enc['attention_mask'] = torch.stack(padded_attn)


def run_one_condition(model, tokenizer, img_proc, data, condition, split_name,
                      t1_idx=None, t2_idx=None, _blank_pix=None):
    """
    Evaluate `data` under one condition.
    condition: 'multimodal' | 'textonly_t1' | 'textonly_t2'
    Manually expands image tokens from 1 to 576 to match vision tower output.
    Returns list of per-sample result dicts.
    """
    records = []
    for item in tqdm(data, desc=f'{condition}|{split_name}', leave=False):
        uid  = item['unique_id']
        name = item['name']
        img_path = Path(FIUBENCH_DIR) / item['image_path']
        if not img_path.exists():
            continue

        real_img = Image.open(img_path).convert('RGB')
        real_pix = img_proc(real_img, return_tensors='pt')['pixel_values'].to(DEVICE, dtype=torch.bfloat16)

        for qi, qa in enumerate(item['qa_list']):
            orig_q     = qa['question']
            answer     = qa['answer']
            keywords   = qa.get('keywords', [])
            para_qs    = qa.get('paraphrased_question', [])
            perturb_as = qa.get('perturbed_answer', [])

            # Select question and pixel values
            if condition == 'multimodal':
                question = orig_q
                pix = real_pix
            elif condition == 'textonly_t1':
                question = t1_idx.get((uid, orig_q.strip().lower()), sub_name(orig_q, name))
                pix = _blank_pix
            elif condition == 'textonly_t2':
                question = t2_idx.get((uid, orig_q.strip().lower()), sub_name(orig_q, name))
                pix = _blank_pix
            else:
                raise ValueError(condition)

            rec = {'uid': uid, 'name': name, 'qi': qi,
                   'question': question, 'answer': answer, 'keywords': keywords}

            try:
                # MINK: forward pass on question + answer
                full_text = phi3_prompt(question) + answer.capitalize()
                enc = tokenizer(full_text, return_tensors='pt', padding=True).to(DEVICE)
                _expand_image_tokens_inplace(enc, 32038)
                labels = enc['input_ids'].clone()
                labels[labels == 32038] = -100
                with torch.no_grad():
                    out = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                                pixel_values=pix, labels=labels)
                rec['mink']    = compute_mink(out.logits, labels[0])
                rec['gt_loss'] = out.loss.item() if out.loss is not None else float('nan')

                # EM: generation
                prompt = phi3_prompt(question)
                enc2 = tokenizer(prompt, return_tensors='pt', padding=True).to(DEVICE)
                _expand_image_tokens_inplace(enc2, 32038)
                with torch.no_grad():
                    gen = model.generate(
                        input_ids=enc2.input_ids, attention_mask=enc2.attention_mask,
                        pixel_values=pix, max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False, pad_token_id=tokenizer.eos_token_id)
                pred = tokenizer.decode(gen[0, enc2.input_ids.shape[-1]:], skip_special_tokens=True).strip()
                rec['pred'] = pred
                rec['em']   = eval_em(pred, answer, keywords)

                # APE: paraphrased question (forget5 only)
                if split_name == 'forget5' and para_qs:
                    pq = para_qs[0] if isinstance(para_qs, list) else para_qs
                    ape_q   = pq if condition == 'multimodal' else sub_name(pq, name)
                    ape_pix = real_pix if condition == 'multimodal' else _blank_pix
                    enc3 = tokenizer(phi3_prompt(ape_q), return_tensors='pt', padding=True).to(DEVICE)
                    _expand_image_tokens_inplace(enc3, 32038)
                    with torch.no_grad():
                        gen3 = model.generate(
                            input_ids=enc3.input_ids, attention_mask=enc3.attention_mask,
                            pixel_values=ape_pix, max_new_tokens=MAX_NEW_TOKENS,
                            do_sample=False, pad_token_id=tokenizer.eos_token_id)
                    pred3 = tokenizer.decode(gen3[0, enc3.input_ids.shape[-1]:], skip_special_tokens=True).strip()
                    rec['ape'] = eval_em(pred3, answer, keywords)
                else:
                    rec['ape'] = None

                # Truth ratio for KS-Test
                if perturb_as and para_qs:
                    pq2   = para_qs[0] if isinstance(para_qs, list) else para_qs
                    tq    = pq2 if condition == 'multimodal' else sub_name(pq2, name)
                    tpix  = real_pix if condition == 'multimodal' else _blank_pix
                    enc_gt = tokenizer(phi3_prompt(tq) + answer.capitalize(),
                                       return_tensors='pt', padding=True).to(DEVICE)
                    _expand_image_tokens_inplace(enc_gt, 32038)
                    labels_gt = enc_gt['input_ids'].clone()
                    labels_gt[labels_gt == 32038] = -100
                    with torch.no_grad():
                        out_gt = model(input_ids=enc_gt.input_ids,
                                       attention_mask=enc_gt.attention_mask,
                                       pixel_values=tpix, labels=labels_gt)
                    glt = out_gt.loss.item() if out_gt.loss is not None else float('nan')

                    ploss = []
                    for pa in perturb_as[:3]:
                        enc_p = tokenizer(phi3_prompt(tq) + pa.capitalize(),
                                          return_tensors='pt', padding=True).to(DEVICE)
                        _expand_image_tokens_inplace(enc_p, 32038)
                        labels_p = enc_p['input_ids'].clone()
                        labels_p[labels_p == 32038] = -100
                        with torch.no_grad():
                            out_p = model(input_ids=enc_p.input_ids,
                                          attention_mask=enc_p.attention_mask,
                                          pixel_values=tpix, labels=labels_p)
                        if out_p.loss is not None:
                            ploss.append(out_p.loss.item())

                    rec['truth_ratio_raw'] = truth_ratio_raw(glt, ploss) if ploss else float('nan')
                else:
                    rec['truth_ratio_raw'] = float('nan')

            except Exception as e:
                import traceback; traceback.print_exc()
                rec.update({'mink': 0.0, 'em': 0.0, 'ape': None,
                             'truth_ratio_raw': float('nan'), 'pred': ''})

            records.append(rec)
    return records


BASE_MODEL_ID = 'xtuner/llava-phi-3-mini-hf'

def _has_weights(directory):
    p = Path(directory)
    return p.exists() and bool(list(p.glob('*.safetensors')) or list(p.glob('pytorch_model*.bin')))

def _load_tokenizer(path):
    """Load tokenizer from path, falling back to base model if tokenizer files are missing."""
    from transformers import AutoTokenizer
    has_tok = (Path(path) / 'tokenizer.json').exists() or (Path(path) / 'tokenizer.model').exists()
    src = path if has_tok else BASE_MODEL_ID
    if src != path:
        print(f'  No tokenizer in {path} — loading from {BASE_MODEL_ID}')
    tok = AutoTokenizer.from_pretrained(src)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def _resolve_model_path(path):
    """Return path if it has model weights, else fall back to base model."""
    if _has_weights(path):
        return path
    print(f'  WARNING: no model weights in {path} — falling back to {BASE_MODEL_ID}')
    return BASE_MODEL_ID


def evaluate_checkpoint(ckpt_name, model_path, img_proc, label):
    """
    Load model, run all conditions, unload. Returns raw results dict.
    stage1: full HF model dir -> from_pretrained directly.
    stage2 (ga/gd/kl/po): LoRA-only checkpoint.pt -> load base=stage1 then apply.
    """
    print(f'\n{"="*60}\nEvaluating: {label}\n{"="*60}')
    if ckpt_name == 'stage1':
        resolved = _resolve_model_path(model_path)
        tokenizer = _load_tokenizer(resolved)
        model = LlavaForConditionalGeneration.from_pretrained(
            resolved, attn_implementation='sdpa', torch_dtype=torch.bfloat16
        ).to(DEVICE)
    else:
        # Stage2: base model is stage1; checkpoint.pt holds only the trained LoRA params
        base_path = _resolve_model_path(STAGE1_LOCAL)
        tokenizer = _load_tokenizer(base_path)
        model = LlavaForConditionalGeneration.from_pretrained(
            base_path, attn_implementation='sdpa', torch_dtype=torch.bfloat16
        ).to(DEVICE)
        lora_state = torch.load(Path(model_path) / 'checkpoint.pt', map_location=DEVICE)
        model.load_state_dict(lora_state, strict=False)
        print(f'  Applied LoRA weights from {model_path}/checkpoint.pt')
    model.eval()
    print(f'  Model loaded on {DEVICE}')

    bp = blank_pixels(img_proc)

    results = {
        'forget_multimodal':  run_one_condition(model, tokenizer, img_proc,
                                                 forget_data, 'multimodal',  'forget5'),
        'forget_textonly_t1': run_one_condition(model, tokenizer, img_proc,
                                                 forget_data, 'textonly_t1', 'forget5',
                                                 t1_index, t2_index, bp),
        'forget_textonly_t2': run_one_condition(model, tokenizer, img_proc,
                                                 forget_data, 'textonly_t2', 'forget5',
                                                 t1_index, t2_index, bp),
        'retain_multimodal':  run_one_condition(model, tokenizer, img_proc,
                                                 retain_data, 'multimodal',  'retain5'),
        'retain_textonly_t1': run_one_condition(model, tokenizer, img_proc,
                                                 retain_data, 'textonly_t1', 'retain5',
                                                 t1_index, t2_index, bp),
    }

    del model
    torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f'{label} done — GPU freed')
    return results

print('evaluate_checkpoint defined')

evaluate_checkpoint defined


## Section 5: Aggregate Metrics & CMLD

In [20]:
from scipy.stats import ks_2samp

def aggregate(forget_recs, retain_recs):
    m = {}
    # EM
    ems = [r['em'] for r in forget_recs if 'em' in r]
    m['em_pct'] = float(np.mean(ems) * 100) if ems else 0.0

    # MINK
    minks = [r['mink'] for r in forget_recs if 'mink' in r]
    m['mink']     = float(np.mean(minks)) if minks else 0.0
    m['mink_pct'] = m['mink'] * 100

    # APE
    apes = [r['ape'] for r in forget_recs if r.get('ape') is not None]
    m['ape_pct'] = float(np.mean(apes) * 100) if apes else None

    # KS-Test
    ftr = np.array([r['truth_ratio_raw'] for r in forget_recs
                    if not (r.get('truth_ratio_raw') is None or
                            (isinstance(r.get('truth_ratio_raw'), float) and math.isnan(r['truth_ratio_raw'])))])
    rtr = np.array([r['truth_ratio_raw'] for r in retain_recs
                    if not (r.get('truth_ratio_raw') is None or
                            (isinstance(r.get('truth_ratio_raw'), float) and math.isnan(r['truth_ratio_raw'])))])
    if len(ftr) > 1 and len(rtr) > 1:
        _, ksp = ks_2samp(ftr, rtr)
        m['ks_p_pct'] = float(ksp * 100)
    else:
        m['ks_p_pct'] = 0.0

    m['compliant'] = m['mink'] <= COMPLIANCE_THRESHOLD
    return m


def compute_all_metrics(results, label):
    mm = aggregate(results['forget_multimodal'],  results['retain_multimodal'])
    t1 = aggregate(results['forget_textonly_t1'], results['retain_textonly_t1'])
    t2 = aggregate(results['forget_textonly_t2'], results['retain_multimodal'])  # retain T2 not needed
    return {
        'label': label,
        'multimodal':  mm,
        'textonly_t1': t1,
        'textonly_t2': t2,
        'cmld_t1': mm['mink'] - t1['mink'],
        'cmld_t2': mm['mink'] - t2['mink'],
    }

def print_summary(m):
    lbl = m['label']
    print(f'\n── {lbl} ──────────────────────────────────────────────────')
    for cond in ['multimodal', 'textonly_t1', 'textonly_t2']:
        c = m[cond]
        ape_s = f"{c['ape_pct']:.1f}%" if c['ape_pct'] is not None else 'N/A'
        ok = 'PASS' if c['compliant'] else 'FAIL'
        print(f'  {cond:<15} EM={c["em_pct"]:5.1f}%  MINK={c["mink_pct"]:5.2f}%  APE={ape_s:<7}  KS-p={c["ks_p_pct"]:5.1f}%  [{ok}]')
    print(f'  CMLD-T1 = {m["cmld_t1"]*100:+.3f}%  |  CMLD-T2 = {m["cmld_t2"]*100:+.3f}%')

print('aggregate / compute_all_metrics defined')

aggregate / compute_all_metrics defined


## Section 6: Run All Checkpoints

In [21]:
import subprocess
result = subprocess.run(["pip", "install", "sentencepiece"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
# ── Output directory ──────────────────────────────────────────────────────────
EXT1_OUT = Path(PROJECT_ROOT) / 'outputs' / 'extension1'
EXT1_OUT.mkdir(parents=True, exist_ok=True)

# ── Checkpoint registry ───────────────────────────────────────────────────────
MODEL_PATHS = {
    'stage1': STAGE1_LOCAL,
    'ga':     METHODS['ga']['local'],
    'gd':     METHODS['gd']['local'],
    'kl':     METHODS['kl']['local'],
    'po':     METHODS['po']['local'],
}

CHECKPOINTS = ['stage1', 'ga', 'gd', 'kl', 'po']

available = {
    'stage1': True,
    'ga':  (Path(METHODS['ga']['local']) / 'checkpoint.pt').exists(),
    'gd':  (Path(METHODS['gd']['local']) / 'checkpoint.pt').exists(),
    'kl':  (Path(METHODS['kl']['local']) / 'checkpoint.pt').exists(),
    'po':  (Path(METHODS['po']['local']) / 'checkpoint.pt').exists(),
}
print('Checkpoint availability:')
for k, v in available.items():
    print(f'  {k:<8} {"available" if v else "MISSING"}')

# ── Image processor ────────────────────────────────────────────────────────────
img_proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-large-patch14-336')

all_raw     = {}
all_metrics = {}

def _nan_clean(obj):
    if isinstance(obj, dict):
        return {k: _nan_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_nan_clean(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

def _cache_is_valid(raw):
    """Return False if every record has mink=0 and pred='' (indicates a failed run)."""
    for split_recs in raw.values():
        for rec in split_recs:
            if rec.get('mink', 0.0) != 0.0 or rec.get('pred', '') != '':
                return True
    return False

# ── Run / load each checkpoint ─────────────────────────────────────────────────
for ckpt_name in CHECKPOINTS:
    if not available.get(ckpt_name):
        print(f'\nSkipping {ckpt_name}: checkpoint.pt not found')
        continue

    cache = EXT1_OUT / f'{ckpt_name}_raw.json'

    if cache.exists():
        cached = json.load(open(cache))
        if _cache_is_valid(cached):
            print(f'\n[{ckpt_name}] Loading cached results from {cache.name}...')
            all_raw[ckpt_name] = cached
        else:
            print(f'\n[{ckpt_name}] Cache contains all-zero results (prior failed run) — recomputing...')
            cache.unlink()

    if ckpt_name not in all_raw:
        raw = evaluate_checkpoint(ckpt_name, MODEL_PATHS[ckpt_name], img_proc, label=ckpt_name)
        all_raw[ckpt_name] = raw
        with open(cache, 'w') as f:
            json.dump(_nan_clean(raw), f)
        print(f'[{ckpt_name}] Saved -> {cache}')

    m = compute_all_metrics(all_raw[ckpt_name], label=ckpt_name)
    all_metrics[ckpt_name] = m
    print_summary(m)

print('\nAll evaluations complete')

Checkpoint availability:
  stage1   available
  ga       available
  gd       available
  kl       available
  po       available

[stage1] Cache contains all-zero results (prior failed run) — recomputing...

Evaluating: stage1


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

  Model loaded on cuda


stage1 done — GPU freed
[stage1] Saved -> /content/FIUBench_Reproducing/outputs/extension1/stage1_raw.json

── stage1 ──────────────────────────────────────────────────
  multimodal      EM= 14.4%  MINK= 0.00%  APE=10.7%    KS-p=  0.8%  [PASS]
  textonly_t1     EM= 21.1%  MINK= 0.00%  APE=16.4%    KS-p=  1.3%  [PASS]
  textonly_t2     EM= 13.0%  MINK= 0.00%  APE=16.4%    KS-p=  9.4%  [PASS]
  CMLD-T1 = -0.000%  |  CMLD-T2 = -0.000%

[ga] Cache contains all-zero results (prior failed run) — recomputing...

Evaluating: ga


Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

  Applied LoRA weights from /content/stage2_ga/checkpoint.pt
  Model loaded on cuda


multimodal|forget5:  25%|██▌       | 5/20 [07:09<21:32, 86.17s/it]

In [13]:
# ── Output directory ──────────────────────────────────────────────────────────
EXT1_OUT = Path(PROJECT_ROOT) / 'outputs' / 'extension1'
EXT1_OUT.mkdir(parents=True, exist_ok=True)

# ── Checkpoint registry ───────────────────────────────────────────────────────
MODEL_PATHS = {
    'stage1': STAGE1_LOCAL,
    'ga':     METHODS['ga']['local'],
    'gd':     METHODS['gd']['local'],
    'kl':     METHODS['kl']['local'],
    'po':     METHODS['po']['local'],
}

CHECKPOINTS = ['stage1', 'ga', 'gd', 'kl', 'po']

# stage1: needs model weights (or falls back to base model)
# stage2: needs checkpoint.pt
available = {
    'stage1': True,  # always attempt; _resolve_model_path handles missing weights
    'ga':  (Path(METHODS['ga']['local']) / 'checkpoint.pt').exists(),
    'gd':  (Path(METHODS['gd']['local']) / 'checkpoint.pt').exists(),
    'kl':  (Path(METHODS['kl']['local']) / 'checkpoint.pt').exists(),
    'po':  (Path(METHODS['po']['local']) / 'checkpoint.pt').exists(),
}
print('Checkpoint availability:')
for k, v in available.items():
    status = 'available' if v else 'MISSING'
    print(f'  {k:<8} {status}')

# ── Image processor ────────────────────────────────────────────────────────────
img_proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-large-patch14-336')

all_raw     = {}
all_metrics = {}

def _nan_clean(obj):
    if isinstance(obj, dict):
        return {k: _nan_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_nan_clean(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

# ── Run / load each checkpoint ─────────────────────────────────────────────────
for ckpt_name in CHECKPOINTS:
    if not available.get(ckpt_name):
        print(f'\nSkipping {ckpt_name}: checkpoint.pt not found')
        continue

    cache = EXT1_OUT / f'{ckpt_name}_raw.json'

    if cache.exists():
        print(f'\n[{ckpt_name}] Loading cached results from {cache.name}...')
        all_raw[ckpt_name] = json.load(open(cache))
    else:
        raw = evaluate_checkpoint(ckpt_name, MODEL_PATHS[ckpt_name], img_proc, label=ckpt_name)
        all_raw[ckpt_name] = raw
        with open(cache, 'w') as f:
            json.dump(_nan_clean(raw), f)
        print(f'[{ckpt_name}] Saved -> {cache}')

    m = compute_all_metrics(all_raw[ckpt_name], label=ckpt_name)
    all_metrics[ckpt_name] = m
    print_summary(m)

print('\nAll evaluations complete')

## Section 7: CMLD Summary Table

In [ ]:
import pandas as pd

METHOD_ORDER  = ['stage1', 'ga', 'gd', 'kl', 'po']
METHOD_LABELS = {'stage1': 'Stage1 (pre)', 'ga': 'GA', 'gd': 'GD', 'kl': 'KL', 'po': 'PO'}

rows = []
for name in METHOD_ORDER:
    if name not in all_metrics:
        continue
    m = all_metrics[name]
    mm, t1 = m['multimodal'], m['textonly_t1']
    rows.append({
        'Method':           METHOD_LABELS[name],
        'MM MINK%':         f"{mm['mink_pct']:.2f}",
        'TO-T1 MINK%':      f"{t1['mink_pct']:.2f}",
        'MM EM%':           f"{mm['em_pct']:.1f}",
        'TO-T1 EM%':        f"{t1['em_pct']:.1f}",
        'MM APE%':          f"{mm['ape_pct']:.1f}" if mm['ape_pct'] is not None else 'N/A',
        'TO-T1 APE%':       f"{t1['ape_pct']:.1f}" if t1['ape_pct'] is not None else 'N/A',
        'CMLD-T1 (pp)':     f"{m['cmld_t1']*100:+.3f}",
        'MM compliant':     'YES' if mm['compliant'] else 'NO',
        'TO-T1 compliant':  'YES' if t1['compliant'] else 'NO',
    })

df = pd.DataFrame(rows)
print('CMLD Summary Table:')
print(df.to_string(index=False))

df.to_csv(EXT1_OUT / 'cmld_summary.csv', index=False)
print(f'\nSaved: {EXT1_OUT}/cmld_summary.csv')

## Section 8: 2×2 Compliance Matrix

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

UNLEARN_METHODS = ['ga', 'gd', 'kl', 'po']
ML = {'ga': 'GA', 'gd': 'GD', 'kl': 'KL', 'po': 'PO'}

# Build quadrant membership
quads = {('pass','pass'):[], ('pass','fail'):[], ('fail','pass'):[], ('fail','fail'):[]}
for name in UNLEARN_METHODS:
    if name not in all_metrics:
        continue
    mm_ok = all_metrics[name]['multimodal']['compliant']
    to_ok = all_metrics[name]['textonly_t1']['compliant']
    quads[('pass' if mm_ok else 'fail', 'pass' if to_ok else 'fail')].append(ML[name])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.set_xlim(0, 2); ax.set_ylim(0, 2)

# Background colors
bg = {(0,0): '#FAEEDD', (1,0): '#D5ECD4', (0,1): '#FFF3CD', (1,1): '#D5ECD4'}
# Semantic: (TO=fail, MM=fail)=red, (TO=pass, MM=pass)=green, (TO=fail, MM=pass)=yellow
bg_fixed = {
    # (xi, yi): xi=0 means TO=fail, xi=1 means TO=pass; yi=0 means MM=fail, yi=1 means MM=pass
    (0,0): '#FAEEDD',   # MM=fail, TO=fail
    (1,0): '#E6F1FB',   # MM=fail, TO=pass  (unusual)
    (0,1): '#FFF3CD',   # MM=pass, TO=fail  (modality-only)
    (1,1): '#D5ECD4',   # MM=pass, TO=pass  (GDPR-compliant)
}
for (xi, yi), col in bg_fixed.items():
    ax.add_patch(mpatches.Rectangle((xi, yi), 1, 1, color=col, zorder=0))

ax.axvline(1, color='black', lw=1.5)
ax.axhline(1, color='black', lw=1.5)

ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(['FAIL', 'PASS'], fontsize=13, fontweight='bold')
ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(['FAIL', 'PASS'], fontsize=13, fontweight='bold')
ax.set_xlabel('Text-Only MIA', fontsize=13)
ax.set_ylabel('Multimodal MIA', fontsize=13)
ax.set_title(f'2×2 Compliance Matrix  (threshold MINK ≤ {COMPLIANCE_THRESHOLD*100:.1f}%)', fontsize=11)

# Quadrant annotation labels
notes = {
    (0.5, 1.5): 'Modality-compliant\nonly',
    (1.5, 1.5): 'GDPR-compliant',
    (0.5, 0.5): 'Not unlearned',
    (1.5, 0.5): 'Unusual',
}
for (x,y), note in notes.items():
    ax.text(x, y-0.27, note, ha='center', va='center', fontsize=8.5, color='gray', style='italic')

# Method names in each quadrant cell
# quads key: (mm_status, to_status); cell position: TO=fail -> xi=0, TO=pass -> xi=1
cell_pos = {
    ('pass','fail'): (0.5, 1.5),
    ('pass','pass'): (1.5, 1.5),
    ('fail','fail'): (0.5, 0.5),
    ('fail','pass'): (1.5, 0.5),
}
for key, pos in cell_pos.items():
    names = quads[key]
    ax.text(pos[0], pos[1]+0.12, ', '.join(names) if names else '—',
            ha='center', va='center', fontsize=15, fontweight='bold')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(EXT1_OUT / f'compliance_matrix.{ext}', dpi=300, bbox_inches='tight')
plt.show()

print('Compliance Matrix:')
for key, names in quads.items():
    print(f'  MM={key[0]}, TO={key[1]}: {names if names else "(none)"}')

## Section 9: CMLD Bar Chart

In [ ]:
avail_methods = [m for m in UNLEARN_METHODS if m in all_metrics]

mm_minks = [all_metrics[m]['multimodal']['mink_pct']  for m in avail_methods]
to_minks = [all_metrics[m]['textonly_t1']['mink_pct'] for m in avail_methods]
cmldt1   = [all_metrics[m]['cmld_t1'] * 100           for m in avail_methods]

x = np.arange(len(avail_methods))
w = 0.32

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w/2, mm_minks, w, label='Post-unlearn MM MINK%',   color='#4C72B0', alpha=0.85)
ax.bar(x + w/2, to_minks, w, label='Post-unlearn TO-T1 MINK%', color='#DD8452', alpha=0.85)
ax.axhline(COMPLIANCE_THRESHOLD*100, color='#2ca02c', linestyle='--', lw=1.8,
           label=f'Compliance threshold ({COMPLIANCE_THRESHOLD*100:.1f}%)')

for i, (c, xp) in enumerate(zip(cmldt1, x)):
    top = max(mm_minks[i], to_minks[i])
    ax.text(xp, top + 0.6, f'CMLD={c:+.2f}pp', ha='center', fontsize=8, color='black')

ax.set_xticks(x)
ax.set_xticklabels([ML[m] for m in avail_methods], fontsize=13)
ax.set_ylabel('MINK Score (%)', fontsize=12)
ax.set_title('Post-Unlearning MINK: Multimodal vs Text-Only\n(lower = better unlearning)', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(mm_minks, default=0), max(to_minks, default=0)) * 1.3 + 1)

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(EXT1_OUT / f'cmld_bar_chart.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {EXT1_OUT}/cmld_bar_chart.*')

## Section 10: Robustness Ablation — T1 vs T2 CMLD Correlation

In [ ]:
from scipy.stats import pearsonr

t1_vals = [all_metrics[m]['cmld_t1'] for m in avail_methods]
t2_vals = [all_metrics[m]['cmld_t2'] for m in avail_methods]

if len(t1_vals) >= 2:
    r, p = pearsonr(t1_vals, t2_vals)
    print(f'Pearson r (CMLD T1 vs T2) = {r:.4f}  (p = {p:.4f})')
    if r > 0.9:
        print('High correlation: CMLD is robust to template phrasing')
    else:
        print('Lower correlation: CMLD may be template-sensitive')
else:
    r = p = None
    print('Need >= 2 methods to compute correlation')

# Scatter plot
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter([v*100 for v in t1_vals], [v*100 for v in t2_vals], s=100, zorder=5, color='#4C72B0')
for i, nm in enumerate(avail_methods):
    ax.annotate(ML[nm], (t1_vals[i]*100, t2_vals[i]*100),
                textcoords='offset points', xytext=(6, 4), fontsize=12)

lim = [min(t1_vals + t2_vals)*100 - 0.2, max(t1_vals + t2_vals)*100 + 0.2]
ax.plot(lim, lim, 'k--', lw=1, alpha=0.4, label='y = x')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.axvline(0, color='gray', lw=0.8, ls=':')

title = 'CMLD Robustness: T1 vs T2'
if r is not None:
    title += f'\nPearson r = {r:.3f}'
ax.set_title(title, fontsize=11)
ax.set_xlabel('CMLD — Template 1 (pp)', fontsize=11)
ax.set_ylabel('CMLD — Template 2 (pp)', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(EXT1_OUT / f'cmld_robustness_scatter.{ext}', dpi=300, bbox_inches='tight')
plt.show()

with open(EXT1_OUT / 'robustness.json', 'w') as f:
    json.dump({
        'pearson_r': float(r) if r else None,
        'pearson_p': float(p) if p else None,
        'cmld_t1_per_method': {ML[m]: float(all_metrics[m]['cmld_t1']) for m in avail_methods},
        'cmld_t2_per_method': {ML[m]: float(all_metrics[m]['cmld_t2']) for m in avail_methods},
    }, f, indent=2)
print(f'Saved: {EXT1_OUT}/robustness.json')

## Section 11: Pre vs Post MINK Comparison Figure

In [ ]:
if 'stage1' in all_metrics and avail_methods:
    s1 = all_metrics['stage1']
    print('Stage1 pre-unlearning baseline:')
    for cond in ['multimodal', 'textonly_t1', 'textonly_t2']:
        print(f'  {cond}: MINK = {s1[cond]["mink_pct"]:.2f}%')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, (cond_key, cond_label) in zip(axes,
            [('multimodal', 'Multimodal'), ('textonly_t1', 'Text-Only T1')]):
        pre = s1[cond_key]['mink_pct']
        post = [all_metrics[m][cond_key]['mink_pct'] for m in avail_methods]
        x = np.arange(len(avail_methods))
        w2 = 0.35
        ax.bar(x - w2/2, [pre]*len(avail_methods), w2, label='Pre-unlearn (Stage1)',
               color='#999999', alpha=0.8)
        ax.bar(x + w2/2, post, w2, label='Post-unlearn',
               color='#4C72B0', alpha=0.85)
        ax.axhline(COMPLIANCE_THRESHOLD*100, color='#2ca02c', linestyle='--',
                   lw=1.5, label=f'Threshold ({COMPLIANCE_THRESHOLD*100:.1f}%)')
        ax.set_xticks(x)
        ax.set_xticklabels([ML[m] for m in avail_methods], fontsize=12)
        ax.set_ylabel('MINK (%)', fontsize=11)
        ax.set_title(f'Pre vs Post MINK — {cond_label}', fontsize=11)
        ax.legend(fontsize=9)
        ax.set_ylim(0, max(pre, max(post, default=0)) * 1.3 + 1)

    plt.tight_layout()
    for ext in ['pdf', 'png']:
        plt.savefig(EXT1_OUT / f'pre_vs_post_mink.{ext}', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {EXT1_OUT}/pre_vs_post_mink.*')
else:
    print('stage1 checkpoint not evaluated — skipping pre vs post comparison')

## Section 12: Save All Metrics & Final Report

In [ ]:
def _ser(obj):
    if isinstance(obj, dict): return {k: _ser(v) for k, v in obj.items()}
    if isinstance(obj, list): return [_ser(v) for v in obj]
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

with open(EXT1_OUT / 'all_metrics.json', 'w') as f:
    json.dump(_ser(all_metrics), f, indent=2)

# Final print
print('=' * 80)
print('EXTENSION 1 — FINAL RESULTS SUMMARY')
print('=' * 80)
print(f'Compliance: MINK <= {COMPLIANCE_THRESHOLD*100:.1f}%  |  CMLD = MM_MINK - TO_MINK  (pp = percentage points)')
print(f'Negative CMLD: text-only leaks more -> cross-modal privacy gap')
print()

header = f'{"Method":<8}  {"Pre-MM":<8}  {"Post-MM":<10}  {"Post-TO":<10}  {"CMLD-T1":<10}  {"MM":<6}  {"TO":<6}  {"GDPR Status"}'
print(header)
print('-' * len(header))

pre_mm = all_metrics.get('stage1', {}).get('multimodal', {}).get('mink_pct', None)
for name in UNLEARN_METHODS:
    if name not in all_metrics: continue
    m = all_metrics[name]
    mm, t1 = m['multimodal'], m['textonly_t1']
    pre_s = f'{pre_mm:.2f}%' if pre_mm else 'N/A'
    gdpr = ('GDPR-compliant' if mm['compliant'] and t1['compliant']
            else 'Modality-only' if mm['compliant']
            else 'Not unlearned')
    print(f'{ML[name]:<8}  {pre_s:<8}  {mm["mink_pct"]:<10.2f}  {t1["mink_pct"]:<10.2f}  {m["cmld_t1"]*100:<+10.3f}  '
          f'{"YES" if mm["compliant"] else "NO":<6}  {"YES" if t1["compliant"] else "NO":<6}  {gdpr}')

print(f'\nFiles in {EXT1_OUT}:')
for p in sorted(EXT1_OUT.iterdir()):
    print(f'  {p.name}')